# 第 31 课｜环境与可复现研究

本 Notebook 对应「365 天量化金融」第 31 课。课程代码主线来自本地视频、课件截图和 Whisper 口播整理；在此基础上补充量化数据语义、断言、性能验证和方法边界。

- 图文口播：`/Users/mojiawen/Documents/学习资料归档/_暂存/pam-video-captures/365-days-quant-finance/31_youtube-QVHRIvr0l18/index.html`
- 环境：`pkm-quant`
- 数据：固定随机种子生成的离线模拟行情（**非真实股票数据**）
- 状态：可运行学习稿；不代表策略收益承诺

> **数据声明**：本 Notebook 的证券名称、价格和成交量均为教学模拟数据。它用于稳定复现计算方法，不能用于评价真实证券、市场规律或策略收益。

## 本节量化问题

课程强调虚拟环境、Jupyter/编辑器双轨制和标准项目目录。本项目已经在 Datawhale 学习阶段建立 `pkm-quant`，因此这里不重复安装，而是验证：

1. Notebook 是否真的使用统一内核；
2. 当前内核实际安装了哪些直接依赖；
3. 研究结果能否追溯到代码、参数、数据快照和运行环境。

### 输入与输出

- 输入：当前 Python 解释器、已安装库、`environment.yml` 和离线行情文件；
- 输出：依赖版本表、环境核验结果、数据文件指纹和研究清单 JSON；
- 验收：内核必须是 `pkm-quant`，课程直接依赖不能缺失，研究清单保存后能原样回读。

## 本节输入是什么？

本节使用三类确定性输入：

1. 手工给出的 3 个价格 `[100.0, 102.0, 101.0]`，只用于演示 pandas 与 NumPy 的最小数据流；
2. `daily_prices.csv` 和 `intraday_5m.csv` 两份**模拟行情文件**，只用于演示数据快照追踪；
3. 当前解释器、已安装包和 `environment.yml`，用于核对运行环境。

模拟行情不对应任何真实证券。本节也不评价策略收益，只回答“这次研究由什么环境、代码和输入文件生成”。

下面的公共准备单元只建立项目路径、绘图字体和随机种子等运行上下文。可核对输出应包括 Python 版本、解释器路径、项目目录和中文字体。

In [1]:
from pathlib import Path
import json
import os
import platform
import sys
import tempfile
import timeit

# Matplotlib 首次运行需要写字体缓存；放到可写临时目录，避免污染用户目录。
MPL_CACHE_DIR = Path(tempfile.gettempdir()) / "pkm-quant-matplotlib"
MPL_CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MPL_CACHE_DIR))

import matplotlib.pyplot as plt
from matplotlib import font_manager
import numpy as np
import pandas as pd
import seaborn as sns

# 从本机字体中选择支持中文的字体，避免标题显示成方框。
preferred_chinese_fonts = [
    "PingFang SC",
    "Arial Unicode MS",
    "Hiragino Sans GB",
    "Heiti SC",
]
available_font_names = {font.name for font in font_manager.fontManager.ttflist}
chinese_font = next(
    (name for name in preferred_chinese_fonts if name in available_font_names),
    "DejaVu Sans",
)
plt.rcParams["font.sans-serif"] = [chinese_font, "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

def find_project_root():
    cwd = Path.cwd().resolve()
    candidates = [
        cwd,
        cwd / "实验" / "p2-python-toolchain",
        cwd / "p2-python-toolchain",
        *cwd.parents,
    ]
    for candidate in candidates:
        if (candidate / "课程辅助设计.md").exists():
            return candidate
    raise FileNotFoundError("请从 p2-python-toolchain 或量化交易项目目录启动 JupyterLab")

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data" / "fixtures"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
RANDOM_SEED = 365

print("Python:", sys.version.split()[0])
print("解释器:", sys.executable)
print("项目:", PROJECT_ROOT)
print("绘图字体:", chinese_font)

Python: 3.11.15
解释器: /opt/homebrew/Caskroom/miniforge/base/envs/pkm-quant/bin/python
项目: /Users/mojiawen/Documents/学习资料归档/_暂存/personal_knowledge_manager/量化交易/实验/p2-python-toolchain
绘图字体: PingFang SC


## 后续课程会用到哪些库？

先建立“库 → 量化任务”的对应关系，不要求一次背完 API。后续课程会在真实场景中逐步展开。

| 库 | 基本对象/能力 | 量化中的主要用途 |
|---|---|---|
| **NumPy** | `ndarray`、广播、矩阵运算 | 收益矩阵、组合加权、向量化计算 |
| **pandas** | `Series`、`DataFrame`、时间索引 | 读取行情、标签对齐、重采样、滚动统计 |
| **SciPy** | 优化、统计分布、数值求解 | 组合优化、参数估计、显著性检验 |
| **Matplotlib / seaborn** | 静态图表和统计图 | K 线以外的收益、回撤、相关性可视化 |
| **scikit-learn** | 特征处理、模型、交叉验证 | 因子/预测模型与样本外验证 |
| **statsmodels** | 回归、时间序列、统计推断 | OLS、协整、残差和置信区间 |
| **yfinance / AkShare** | 数据接口 | 分别接入海外和国内市场数据；本批先用离线数据 |

学习重点不是孤立记忆库名，而是知道：数据先由 pandas 按日期和证券标签对齐，再在必要处转换为 NumPy 做批量计算，最后用统计库和图表库验证结果。

## 最小数据流：先展示输入价格

先建立一张只有 3 行的带标签价格表。此时还没有计算收益，输出中的一行只代表一个观察日的收盘价。

In [2]:
# 输入是 3 个确定价格；索引和列名保留观察语义。
demo_prices = pd.DataFrame(
    {"close": [100.0, 102.0, 101.0]},  # [100.0, 102.0, 101.0]
    index=pd.Index(["第 1 日", "第 2 日", "第 3 日"], name="观察日"),
)

assert demo_prices.shape == (3, 1)
demo_prices

,close
观察日,
第 1 日,100.0
第 2 日,102.0
第 3 日,101.0


## 从 pandas 收益表转成 NumPy 数组

下面的数据流会在后续课程反复出现：

1. `pd.DataFrame` 保存带列名的价格表；
2. `pct_change(fill_method=None)` 按列计算简单收益率 `本期价格 / 上期价格 - 1`；
3. `dropna()` 去掉没有前值的首行；
4. `to_numpy(dtype=np.float64)` 取出数值，交给 NumPy 计算；
5. `np.mean` 得到这两个有效观测的样本算术平均日收益。

`to_numpy()` 后不再保留“观察日”和“close”标签，因此应先由 pandas 完成对齐，并在输出时恢复或明确记录标签。这里的样本只有两期有效收益，均值只用于核对 API，不是未来预期收益。

In [3]:
# 先保留带标签收益表，再转换为包含两个有效收益的 NumPy 数组。
demo_returns = demo_prices.copy()
demo_returns["simple_return"] = demo_returns["close"].pct_change(
    fill_method=None
)
return_array = (
    demo_returns["simple_return"]
    .dropna()
    .to_numpy(dtype=np.float64)
)
sample_mean_return = float(np.mean(return_array))

expected_returns = np.array([0.02, 101.0 / 102.0 - 1])
np.testing.assert_allclose(return_array, expected_returns)
assert return_array.shape == (2,)
assert np.isclose(sample_mean_return, 0.0050980392156863)

display(demo_returns.round(6))
pd.DataFrame(
    {
        "核对结果": [
            str(return_array.shape),
            f"{return_array[0]:.4%}",
            f"{return_array[1]:.4%}",
            f"{sample_mean_return:.4%}",
        ]
    },
    index=[
        "有效收益数组 shape",
        "第 2 日简单收益",
        "第 3 日简单收益",
        "两期样本算术平均日收益",
    ],
)

,close,simple_return
观察日,,
第 1 日,100.0,NaN
第 2 日,102.0,0.020000
第 3 日,101.0,-0.009804


,核对结果
有效收益数组 shape,"(2,)"
第 2 日简单收益,2.0000%
第 3 日简单收益,-0.9804%
两期样本算术平均日收益,0.5098%


## 核对实际安装的依赖版本

`importlib.metadata.version(distribution)` 查询当前解释器环境中已安装的**分发包元数据**。例如 `scikit-learn` 是分发包名，Python 中通常使用 `import sklearn`。

捕获 `PackageNotFoundError` 后标记为 `MISSING`。环境文件表达“计划安装什么”，版本表表达“当前环境登记了什么版本”；版本存在不等于所有 API、原生依赖或网络数据接口都已成功运行，因此后续课程仍要执行实际任务验收。

In [4]:
# 查询课程直接依赖的安装版本，并显式标记是否存在。
from importlib import metadata

packages = [
    "numpy", "pandas", "scipy", "matplotlib", "seaborn",
    "scikit-learn", "statsmodels", "jupyterlab", "ipykernel",
    "yfinance", "akshare",
]
versions = {}
for package in packages:
    try:
        versions[package] = metadata.version(package)
    except metadata.PackageNotFoundError:
        versions[package] = "MISSING"

environment_table = pd.DataFrame(
    {
        "version": pd.Series(versions),
        "installed": pd.Series(versions).ne("MISSING"),
    }
)
environment_table.index.name = "distribution"
missing_packages = environment_table.index[
    ~environment_table["installed"]
].tolist()

assert not missing_packages, f"缺少课程依赖：{missing_packages}"
environment_table

,version,installed
distribution,,
numpy,2.4.6,True
pandas,3.0.3,True
scipy,1.17.1,True
matplotlib,3.11.0,True
seaborn,0.13.2,True
scikit-learn,1.9.0,True
statsmodels,0.14.6,True
jupyterlab,4.6.1,True
ipykernel,7.3.0,True


## 核对当前 Notebook 的实际解释器

`sys.executable` 是当前 Notebook 实际调用的 Python 路径，不是终端中另一个 Python 的路径。这里从解释器所在目录读取环境名，并用带标签表格同时展示实际值和期望值。

断言失败会立即终止执行，避免在错误内核中继续产生看似正常的研究结果。

In [5]:
# 核对 Notebook 内核实际使用的解释器，而不是只看界面显示名称。
current_env_name = Path(sys.executable).parent.parent.name
kernel_check = pd.DataFrame(
    {
        "实际值": [
            sys.version.split()[0],
            sys.executable,
            current_env_name,
            len(missing_packages),
        ],
        "期望值": [
            "3.11.x",
            "pkm-quant 环境中的 python",
            "pkm-quant",
            0,
        ],
    },
    index=[
        "Python 版本",
        "解释器路径",
        "Conda 环境名",
        "缺失直接依赖数量",
    ],
)

assert current_env_name == "pkm-quant", (
    "当前 Notebook 没有使用 pkm-quant 解释器"
)
kernel_check

,实际值,期望值
Python 版本,3.11.15,3.11.x
解释器路径,/opt/homebrew/Caskroom/miniforge/base/envs/pkm...,pkm-quant 环境中的 python
Conda 环境名,pkm-quant,pkm-quant
缺失直接依赖数量,0,0


## 核对 `environment.yml` 的环境声明

`environment.yml` 是声明直接依赖的环境规格文件；它适合回答“项目计划安装什么”，但没有把每个传递依赖精确锁到唯一构建，因此不能替代锁文件或容器镜像。

下面只提取可核对的关键信息，不截断打印整份 YAML。

In [6]:
# 读取环境规格，并提取环境名与关键直接依赖核对。
environment_file = PROJECT_ROOT.parents[1] / "environment.yml"
environment_text = environment_file.read_text(encoding="utf-8")

declared_env_name = None
for line in environment_text.splitlines():
    if line.startswith("name:"):
        declared_env_name = line.split(":", 1)[1].strip()
        break

environment_declaration = pd.DataFrame(
    {
        "实际值": [
            str(environment_file),
            declared_env_name,
            "  - numpy" in environment_text,
            "  - pandas" in environment_text,
            len(environment_text.splitlines()),
        ]
    },
    index=[
        "环境文件",
        "声明环境名",
        "声明 NumPy",
        "声明 pandas",
        "文件行数",
    ],
)

assert declared_env_name == "pkm-quant"
assert "  - numpy" in environment_text
assert "  - pandas" in environment_text
environment_declaration

,实际值
环境文件,/Users/mojiawen/Documents/学习资料归档/_暂存/personal_...
声明环境名,pkm-quant
声明 NumPy,True
声明 pandas,True
文件行数,39


## 研究清单：给实验建立可追溯身份

下面的代码**不负责选股或计算收益**，而是生成一张研究清单（research manifest）。

一份回测结果并不只由代码决定：

> 回测结果 = 代码 × 参数 × 数据快照 × 运行环境

同一段代码如果换了行情文件、随机种子或库版本，可能产生不同结果。为了以后回答“这份结果由什么生成”，本节清单记录：

| 清单字段 | 记录什么 | 量化研究中的用途 |
|---|---|---|
| `python` / `executable` | Python 版本和实际解释器 | 确认没有误用其他 Conda 环境 |
| `packages` | NumPy、pandas 等分发包版本 | 排查 API 或计算行为变化 |
| `random_seed` | 本项目使用的确定性随机种子 | 复现模拟数据与抽样 |
| `code` | 当前 Notebook 生成脚本的 SHA-256 摘要 | 判断生成实现的文件字节是否变化 |
| `data` | 每个输入 CSV 的 SHA-256 摘要 | 判断输入文件字节是否变化 |

### 第一步：先展示将要追踪的输入文件

研究清单不能只出现一串哈希值。先查看每个 CSV 的行数、列数、时间字段、首尾时间和文件大小，确认“本次究竟追踪了什么输入”。

In [7]:
# 先读取两份小型教学 fixture，建立带标签的输入文件清单。
fixture_files = sorted(DATA_DIR.glob("*.csv"))
assert fixture_files, "没有找到待追踪的 CSV 文件"
inventory_rows = []
schema_rows = []

for path in fixture_files:
    frame = pd.read_csv(path)
    if "date" in frame.columns:
        time_field = "date"
    elif "timestamp" in frame.columns:
        time_field = "timestamp"
    else:
        time_field = None

    inventory_rows.append(
        {
            "file": path.name,
            "rows": len(frame),
            "columns": frame.shape[1],
            "time_field": time_field,
            "first_time": (
                frame[time_field].iloc[0] if time_field else "N/A"
            ),
            "last_time": (
                frame[time_field].iloc[-1] if time_field else "N/A"
            ),
            "bytes": path.stat().st_size,
        }
    )
    schema_rows.append(
        {
            "file": path.name,
            "fields": ", ".join(frame.columns),
        }
    )

file_inventory = pd.DataFrame(inventory_rows).set_index("file")
schema_table = pd.DataFrame(schema_rows).set_index("file")
assert not file_inventory.empty
display(file_inventory)
schema_table

,rows,columns,time_field,first_time,last_time,bytes
file,,,,,,
daily_prices.csv,4536,9,date,2023-01-03,2025-11-25,330213
intraday_5m.csv,480,8,timestamp,2025-03-03T09:35:00+08:00,2025-03-14T15:00:00+08:00,41478


,fields
file,
daily_prices.csv,"date, symbol, sector, open, high, low, close, ..."
intraday_5m.csv,"timestamp, symbol, open, high, low, close, vol..."


### 第二步：计算并展示 SHA-256 摘要

`hashlib.sha256()` 创建摘要计算器；文件以二进制模式分块读取，每次读取 `1 MiB`，因此大文件也不必一次载入内存。表格只展示摘要前 12 位便于阅读，完整 64 位摘要仍保存在清单中。

SHA-256 比较的是**文件字节**：同一逻辑表若改变列顺序、换行符或保存格式，摘要也会变化。摘要可以高概率识别内容变化，但不能证明行情来源可信、复权正确或不存在未来数据。

In [8]:
import hashlib

# 为每个输入 CSV 分块计算 SHA-256 内容摘要。
def file_sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while True:
            chunk = handle.read(1024 * 1024)  # 每次读取 1 MiB
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()

data_fingerprints = {}
for path in fixture_files:
    data_fingerprints[path.name] = file_sha256(path)

# 表格只展示短前缀；data_fingerprints 保留完整 64 位摘要。
fingerprint_table = pd.DataFrame(
    {
        "file": [path.name for path in fixture_files],
        "bytes": [path.stat().st_size for path in fixture_files],
        "sha256_prefix": [
            data_fingerprints[path.name][:12] for path in fixture_files
        ],
    }
).set_index("file")

assert all(len(value) == 64 for value in data_fingerprints.values())
fingerprint_table

,bytes,sha256_prefix
file,,
daily_prices.csv,330213,e57dd4377afd
intraday_5m.csv,41478,c54983804096


### 第三步：构造并展示研究清单

研究清单是普通 Python 字典，集中记录解释器、平台、随机种子、依赖版本、当前生成脚本和完整数据摘要。

代码摘要的键使用相对路径 `tools/build_first_batch.py`，避免只保存同名文件名而无法判断它位于项目何处。这个摘要覆盖当前生成脚本文件，不自动覆盖手工修改的 Notebook、其他仓库代码或外部服务状态。

In [9]:
# 将环境、参数、代码快照和数据快照组成研究清单。
generator_path = PROJECT_ROOT / "tools" / "build_first_batch.py"
generator_key = generator_path.relative_to(PROJECT_ROOT).as_posix()
generator_fingerprint = file_sha256(generator_path)

manifest = {
    "python": sys.version.split()[0],
    "executable": sys.executable,
    "platform": platform.platform(),
    "random_seed": RANDOM_SEED,
    "packages": versions,
    "code": {generator_key: generator_fingerprint},
    "data": data_fingerprints,
}

manifest_summary = pd.DataFrame(
    {
        "值": [
            manifest["python"],
            current_env_name,
            manifest["random_seed"],
            len(manifest["packages"]),
            generator_key,
            generator_fingerprint[:12],
            len(manifest["data"]),
        ]
    },
    index=[
        "Python",
        "环境",
        "随机种子",
        "分发包数量",
        "代码文件",
        "代码 SHA-256 前缀",
        "数据文件数量",
    ],
)
manifest_summary

,值
Python,3.11.15
环境,pkm-quant
随机种子,365
分发包数量,11
代码文件,tools/build_first_batch.py
代码 SHA-256 前缀,d5679c99e535
数据文件数量,2


### 第四步：保存、回读并核对 JSON

`json.dumps(..., ensure_ascii=False, indent=2)` 生成可读的 UTF-8 JSON；`Path.write_text` 保存，`json.loads` 回读。写完立即回读并断言关键字段，是最小的交付校验。

In [10]:
# 保存研究清单，再回读验证关键字段。
manifest_path = OUTPUT_DIR / "31_research_manifest.json"
manifest_path.write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)
manifest_reloaded = json.loads(manifest_path.read_text(encoding="utf-8"))
assert manifest_reloaded["random_seed"] == RANDOM_SEED
assert manifest_reloaded["code"] == manifest["code"]
assert manifest_reloaded["data"] == data_fingerprints
assert manifest_reloaded == manifest

pd.DataFrame(
    {
        "核对结果": [
            manifest_path.exists(),
            manifest_reloaded == manifest,
            len(next(iter(manifest_reloaded["code"].values()))),
            len(manifest_reloaded["data"]),
            str(manifest_path),
        ]
    },
    index=[
        "JSON 文件存在",
        "回读内容与内存对象一致",
        "代码摘要字符数",
        "数据文件数量",
        "输出路径",
    ],
)

,核对结果
JSON 文件存在,True
回读内容与内存对象一致,True
代码摘要字符数,64
数据文件数量,2
输出路径,/Users/mojiawen/Documents/学习资料归档/_暂存/personal_...


## 可复用结论

重点不是背 `hashlib` 的写法，而是建立四个研究习惯：

1. 计算前先展示输入，明确文件、shape、时间范围和字段列表；
2. 结果同时记录解释器、直接依赖、随机种子、代码文件和数据文件；
3. 文件摘要变化时，前后两次结果不能视为完全相同的实验；
4. 可追溯性只回答“由什么生成”，正确性仍需数据校验、断言和样本外检验。

## 方法边界

- 镜像源只影响下载位置，不保证代码兼容。
- `environment.yml` 适合声明直接依赖；生产复现还需锁文件或容器。
- 固定随机种子只控制显式使用它的随机过程；并行计算、硬件和外部接口仍可能带来差异。
- SHA-256 不能证明数据真实或未被恶意替换；可信来源、签名和业务校验仍需单独记录。
- 当前 `code` 字段只追踪生成脚本，不代表项目全部源码、Notebook 手工修改和 Git 状态。
- Notebook 用于探索，重复逻辑应进入 `src/` 并由测试保护。
- 本节验收是“环境可追溯”，不是“重新安装一次 Anaconda”。

## 练习

1. 把最小价格例子的第 3 日价格从 `101.0` 改为 `103.0`，先手算第 3 日收益，再核对 NumPy 输出和样本均值。
2. 复制一份教学 CSV 到临时目录，只改变一个字符，比较修改前后的 SHA-256；不要修改正式 fixture。
3. 假设数据摘要不变、代码摘要变化，写出可以下结论和不能下结论的内容。
4. 查看 `environment.yml`，解释为什么“环境名相同”仍不足以证明两台机器的所有传递依赖完全一致。